In [1]:
import pandas as pd
import sqlite3

In [2]:
# Load the Orders and Returns sheets from your Excel file
orders = pd.read_excel("Global Superstore Data.xlsx", sheet_name="Orders")
returns = pd.read_excel("Global Superstore Data.xlsx", sheet_name="Returns")

print(orders.shape)   # sanity check: should show (51290, 24)
print(orders.head())

(51290, 24)
   Row ID                Order ID Order Date  Ship Date       Ship Mode  \
0   24599  IN-2017-CA120551-42816 2017-03-22 2017-03-29  Standard Class   
1   29465  ID-2015-BD116051-42248 2015-09-01 2015-09-04    Second Class   
2   24598  IN-2017-CA120551-42816 2017-03-22 2017-03-29  Standard Class   
3   24597  IN-2017-CA120551-42816 2017-03-22 2017-03-29  Standard Class   
4   29464  ID-2015-BD116051-42248 2015-09-01 2015-09-04    Second Class   

  Customer ID    Customer Name      Segment  Postal Code   City  ...  \
0   CA-120551  Cathy Armstrong  Home Office          NaN  Herat  ...   
1   BD-116051     Brian Dahlen     Consumer          NaN  Herat  ...   
2   CA-120551  Cathy Armstrong  Home Office          NaN  Herat  ...   
3   CA-120551  Cathy Armstrong  Home Office          NaN  Herat  ...   
4   BD-116051     Brian Dahlen     Consumer          NaN  Herat  ...   

    Product ID                                       Product Name  \
0  FUR-BO-4861                    I

In [3]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"])

In [4]:
daily = orders.groupby(orders["Order Date"].dt.date).agg(
    Revenue=("Sales", "sum"),
    Orders=("Order ID", "nunique")
).reset_index()

daily["AOV"] = daily["Revenue"] / daily["Orders"]

daily = daily.rename(columns={"Order Date": "Date"})

print(daily.shape)
daily.head()

(1429, 4)


,Date,Revenue,Orders,AOV
0,2014-01-01,808.56300,4,202.140750
1,2014-01-02,314.22000,1,314.220000
2,2014-01-03,4503.53720,11,409.412473
3,2014-01-04,2808.87024,10,280.887024
4,2014-01-05,3662.31000,4,915.577500


In [5]:
# Get the Order Date for each returned Order ID
returns_with_dates = returns.merge(
    orders[["Order ID", "Order Date"]].drop_duplicates(),
    on="Order ID",
    how="left"
)

returns_with_dates["Order Date"] = returns_with_dates["Order Date"].dt.date

# Count returned orders per day
returns_daily = returns_with_dates.groupby("Order Date").agg(
    ReturnedOrders=("Order ID", "nunique")
).reset_index()

returns_daily = returns_daily.rename(columns={"Order Date": "Date"})

print(returns_daily.shape)
returns_daily.head()

(937, 2)


,Date,ReturnedOrders
0,2014-01-04,2
1,2014-01-07,1
2,2014-01-10,2
3,2014-01-11,1
4,2014-01-14,1


In [6]:
daily = daily.merge(returns_daily, on="Date", how="left")
daily["ReturnedOrders"] = daily["ReturnedOrders"].fillna(0)
daily["ReturnRate"] = daily["ReturnedOrders"] / daily["Orders"]

daily.head(10)

,Date,Revenue,Orders,AOV,ReturnedOrders,ReturnRate
0,2014-01-01,808.56300,4,202.140750,0.0,0.000000
1,2014-01-02,314.22000,1,314.220000,0.0,0.000000
2,2014-01-03,4503.53720,11,409.412473,0.0,0.000000
3,2014-01-04,2808.87024,10,280.887024,2.0,0.200000
4,2014-01-05,3662.31000,4,915.577500,0.0,0.000000
5,2014-01-06,622.53810,6,103.756350,0.0,0.000000
6,2014-01-07,7123.01850,11,647.547136,1.0,0.090909
7,2014-01-08,6293.26000,6,1048.876667,0.0,0.000000
8,2014-01-09,813.74940,3,271.249800,0.0,0.000000
9,2014-01-10,6794.18116,11,617.652833,2.0,0.181818


In [7]:
# Connect to (or create) a SQLite database file
conn = sqlite3.connect("superstore.db")

# Write the daily table into it as a table called 'daily_kpis'
daily.to_sql("daily_kpis", conn, if_exists="replace", index=False)

conn.commit()

In [8]:
check = pd.read_sql("SELECT * FROM daily_kpis LIMIT 10", conn)
check

,Date,Revenue,Orders,AOV,ReturnedOrders,ReturnRate
0,2014-01-01,808.56300,4,202.140750,0.0,0.000000
1,2014-01-02,314.22000,1,314.220000,0.0,0.000000
2,2014-01-03,4503.53720,11,409.412473,0.0,0.000000
3,2014-01-04,2808.87024,10,280.887024,2.0,0.200000
4,2014-01-05,3662.31000,4,915.577500,0.0,0.000000
5,2014-01-06,622.53810,6,103.756350,0.0,0.000000
6,2014-01-07,7123.01850,11,647.547136,1.0,0.090909
7,2014-01-08,6293.26000,6,1048.876667,0.0,0.000000
8,2014-01-09,813.74940,3,271.249800,0.0,0.000000
9,2014-01-10,6794.18116,11,617.652833,2.0,0.181818


In [9]:
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values("Date").reset_index(drop=True)

In [10]:
kpi_cols = ["Revenue", "Orders", "AOV", "ReturnRate"]

for col in kpi_cols:
    roll_mean = daily[col].rolling(window=30, min_periods=10).mean()
    roll_std = daily[col].rolling(window=30, min_periods=10).std()

    daily[f"{col}_zscore"] = (daily[col] - roll_mean) / roll_std
    daily[f"{col}_anomaly_z"] = daily[f"{col}_zscore"].abs() > 2

In [11]:
for col in kpi_cols:
    roll_q1 = daily[col].rolling(window=30, min_periods=10).quantile(0.25)
    roll_q3 = daily[col].rolling(window=30, min_periods=10).quantile(0.75)
    roll_iqr = roll_q3 - roll_q1

    lower_bound = roll_q1 - 1.5 * roll_iqr
    upper_bound = roll_q3 + 1.5 * roll_iqr

    daily[f"{col}_anomaly_iqr"] = (daily[col] < lower_bound) | (daily[col] > upper_bound)

In [12]:
daily["is_anomaly"] = daily[[f"{c}_anomaly_z" for c in kpi_cols]].any(axis=1) | \
                       daily[[f"{c}_anomaly_iqr" for c in kpi_cols]].any(axis=1)

daily["is_anomaly"].sum()   # how many anomaly days were flagged

np.int64(191)

In [14]:
daily.to_sql("daily_kpis", conn, if_exists="replace", index=False)
conn.commit()